In [1]:
pip install fastapi # This line performs: pip install fastapi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.8 MB/s eta 0:00:00


In [2]:
pip install uvicorn # This line performs: pip install uvicorn...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.0 MB/s eta 0:00:00


In [3]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score, precision_score
import fastapi
from typing import Optional

In [4]:


_x = [1]
_y = [1]
_prediction = ["True"]
_result = ["True"]
model = DummyClassifier(strategy="constant", constant="True")
model.fit(np.array([_x,_y]).reshape(1, -1), _result)


DummyClassifier(constant='True', strategy='constant')

In [5]:
def model_evaluation(y_test, y_pred) :
    print('Trained Model Test Data Accuracy Score :',accuracy_score(y_test, y_pred)*100)
    testacc=accuracy_score(y_test, y_pred)
    testrecall=recall_score(y_test, y_pred, pos_label = 'True')
    testprecision=precision_score(y_test, y_pred, pos_label = 'True')
    print(' ')
    print(classification_report(y_test, y_pred))
    return testacc, testrecall, testprecision

In [6]:
def create_eval_df(x, y, result) :
    data = {"x": x, "y": y, "result": result}
    df = pd.DataFrame(data)
    eval_df = df.copy()
    eval_df["predict"] = model.predict(data[x,y])
    eval_df["correct"] = eval_df["predict"] == eval_df["result"]
    eval_df["correctandtrue"] = eval_df["predict"].isin([True]) & eval_df["result"].isin([True])
    return eval_df

In [7]:
app = fastapi.FastAPI()


@app.get("/predict/")
def predict(id:int = 0 , x:float =0, y:float=0):
    print(id, x, y)
    _x.insert(id, x)
    _y.insert(id, y)
    prediction = model.predict(np.array([x,y]).reshape(1, -1))
    _prediction.insert(id, prediction[0])
    print(prediction[0])
    return dict([("prediction", prediction[0])])

@app.get("/result/")
def result(id:int, result:str):
    print(result)
    _result.insert(id, result)
    return dict([("message", "Copied result for {}".format(id))])

@app.get("/retrain/")
def retrain(last_n:int):
    print ("RETRAINING over last {}".format(last_n))
    #print(_x)
    #print(_y)
    n = min(last_n, len(_x) - 1)

    # Get the arrays for training
    X = np.array([_x, _y]).T
    Y = np.array(_result)


    X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.40)

    print(X_train)
    print(y_train)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    #print(X, Y)
    X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.40)

    print(X_train)
    print(y_train)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    testacc, testrecall, testprecision = model_evaluation(y_test, y_pred)

    return {"message": "Model retrained", "accuracy": testacc, "recall": testrecall, "precision": testprecision}

@app.get("/new_model/")
def new_model(model:str, last_n:int, strategy:Optional[str] = None, k:Optional[int] = 0):
    print(model, last_n, strategy, k)
    if model == "Dummy" :
        model = DummyClassifier(strategy=strategy)
    elif model == "KNN" :
        model = KNeighborsClassifier(k=k)
    #model.set_params(model_params)

    message = retrain(last_n=last_n)

    returns = {"message": "Model trained {}".format(model), "train": message}
    print(returns)

    return returns

#@app.get("/evaluate_model/")
#def evaluate_model(last_n:int):
#    return {"message": "Item created", "last_n": last_n}

In [8]:
#print(predict(1, 1, 1))
#result(1, True)
# Populate with synthetic data (add this before calling new_model)
for i in range(50):
    _x.append(i)
    _y.append(i % 2)
    _result.append(i % 3 == 0)
new_model("Dummy", 50, "stratified")

Dummy 50 stratified 0
RETRAINING over last 50
[[19  1]
 [39  1]
 [17  1]
 [28  0]
 [ 6  0]
 [16  0]
 [22  0]
 [ 9  1]
 [12  0]
 [18  0]
 [37  1]
 [23  1]
 [47  1]
 [27  1]
 [33  1]
 [49  1]
 [45  1]
 [ 1  1]
 [13  1]
 [14  0]
 [11  1]
 [ 2  0]
 [32  0]
 [35  1]
 [29  1]
 [31  1]
 [36  0]
 [ 7  1]
 [30  0]
 [34  0]]
['False' 'True' 'False' 'False' 'True' 'False' 'False' 'True' 'True'
 'True' 'False' 'False' 'False' 'True' 'True' 'False' 'True' 'False'
 'False' 'False' 'False' 'False' 'False' 'False' 'False' 'False' 'True'
 'False' 'True' 'False']
[[40  0]
 [48  0]
 [45  1]
 [49  1]
 [21  1]
 [ 6  0]
 [44  0]
 [ 5  1]
 [30  0]
 [10  0]
 [25  1]
 [42  0]
 [15  1]
 [ 1  1]
 [12  0]
 [38  0]
 [46  0]
 [34  0]
 [22  0]
 [32  0]
 [20  0]
 [36  0]
 [35  1]
 [17  1]
 [ 3  1]
 [ 8  0]
 [13  1]
 [11  1]
 [ 2  0]
 [ 1  1]]
['False' 'True' 'True' 'False' 'True' 'True' 'False' 'False' 'True'
 'False' 'False' 'True' 'True' 'True' 'True' 'False' 'False' 'False'
 'False' 'False' 'False' 'True' 'False' 

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'message': "Model trained DummyClassifier(strategy='stratified')",
 'train': {'message': 'Model retrained',
  'accuracy': 0.3333333333333333,
  'recall': 1.0,
  'precision': 0.3333333333333333}}

In [9]:
@app.get("/items/")
async def read_item(skip: int = 0, limit: int = 10):
    return {"skip": skip, "limit": limit}

In [10]:
import asyncio
import uvicorn

if __name__ == "__main__":
    config = uvicorn.Config(app)
    server = uvicorn.Server(config)
    loop = asyncio.get_running_loop()
    loop.create_task(server.serve())